# 📈 Task 5 & 6: Feature Importance & Linear Regression from Scratch

---

## 🎯 Objectives
1. Visualize feature importance to understand what drives predictions
2. Build Linear Regression from scratch using only NumPy
3. Implement Gradient Descent optimization
4. Evaluate model performance

---

## 🔬 Mathematical Foundation: Linear Regression

### 1. The Linear Model (Hypothesis)

Linear regression models the relationship between features $X$ and target $y$:

$$h_{\theta}(X) = \theta_0 + \theta_1 x_1 + \theta_2 x_2 + ... + \theta_n x_n = \theta^T X$$

In matrix form:
$$\hat{y} = X \cdot \theta$$

Where:
- $X$ = Feature matrix $(m \times n)$, m samples, n features
- $\theta$ = Weight vector $(n \times 1)$
- $\hat{y}$ = Predictions $(m \times 1)$

---

### 2. Cost Function (Mean Squared Error)

Measures how wrong our predictions are:

$$J(\theta) = \frac{1}{2m} \sum_{i=1}^{m} (h_{\theta}(x^{(i)}) - y^{(i)})^2$$

In matrix form:
$$J(\theta) = \frac{1}{2m} (X\theta - y)^T (X\theta - y)$$

**Goal:** Find $\theta$ that minimizes $J(\theta)$

---

### 3. Gradient Descent Algorithm

Iteratively update weights to minimize cost:

$$\theta_j := \theta_j - \alpha \frac{\partial J(\theta)}{\partial \theta_j}$$

The gradient (partial derivative):
$$\frac{\partial J(\theta)}{\partial \theta_j} = \frac{1}{m} \sum_{i=1}^{m} (h_{\theta}(x^{(i)}) - y^{(i)}) \cdot x_j^{(i)}$$

In matrix form:
$$\nabla J(\theta) = \frac{1}{m} X^T (X\theta - y)$$

Update rule:
$$\theta := \theta - \alpha \cdot \nabla J(\theta)$$

Where $\alpha$ is the **learning rate** (typically 0.01 to 0.1)

---

### 4. Normal Equation (Closed-Form Solution)

Alternative to gradient descent - direct solution:

$$\theta = (X^T X)^{-1} X^T y$$

**Pros:** No iterations, no learning rate
**Cons:** Slow for large n (matrix inversion is $O(n^3)$)

---

### 5. Evaluation Metrics

**Mean Squared Error (MSE):**
$$MSE = \frac{1}{m} \sum_{i=1}^{m} (y_i - \hat{y}_i)^2$$

**Mean Absolute Error (MAE):**
$$MAE = \frac{1}{m} \sum_{i=1}^{m} |y_i - \hat{y}_i|$$

**R² Score (Coefficient of Determination):**
$$R^2 = 1 - \frac{SS_{res}}{SS_{tot}} = 1 - \frac{\sum(y_i - \hat{y}_i)^2}{\sum(y_i - \bar{y})^2}$$

- $R^2 = 1$: Perfect fit
- $R^2 = 0$: Model predicts mean
- $R^2 < 0$: Worse than mean

---

## 🔄 Gradient Descent Visualization

```
Cost J(θ)
    │
    │     ╭──────╮
    │    ╱        ╲
    │   ╱   Start  ╲         ← Initial random weights
    │  ╱      ●     ╲
    │ ╱        ↘     ╲
    │╱          ↘     ╲
    │            ●     │    ← Step 1 (follow gradient)
    │             ↘    │
    │              ●   │    ← Step 2
    │               ↘  │
    │           Minimum●    ← Optimal θ (convergence)
    └───────────────────────── θ
```

## 📦 Import Libraries & Load Data

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import json
import warnings

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
np.random.seed(42)

print("Libraries loaded successfully!")

In [ ]:
# Load feature-engineered data
df = pd.read_csv('../data/processed/uav_telemetry_features.csv')

print(f"Dataset loaded: {df.shape[0]} samples, {df.shape[1]} features")
df.head()

## 📊 Task 5: Feature Importance Visualization

In [ ]:
def pearson_correlation(x, y):
    """Calculate Pearson correlation from scratch."""
    x, y = np.array(x), np.array(y)
    x_mean, y_mean = np.mean(x), np.mean(y)
    numerator = np.sum((x - x_mean) * (y - y_mean))
    denominator = np.sqrt(np.sum((x - x_mean)**2) * np.sum((y - y_mean)**2))
    return numerator / denominator if denominator != 0 else 0


# Define features and target
feature_columns = [
    'battery_soc', 'voltage', 'current', 'battery_temperature',
    'wind_speed', 'dust_level', 'ambient_temperature', 'altitude',
    'payload_weight', 'flight_speed', 'power_consumption'
]

# Add engineered features if present
engineered = ['available_energy_wh', 'theoretical_flight_time', 'environmental_stress',
              'power_loading', 'soc_power_ratio', 'wind_speed_squared']
for col in engineered:
    if col in df.columns:
        feature_columns.append(col)

target = 'max_range'

# Calculate feature importance (correlation)
importance = {}
for col in feature_columns:
    importance[col] = pearson_correlation(df[col], df[target])

# Sort by absolute importance
sorted_importance = sorted(importance.items(), key=lambda x: abs(x[1]), reverse=True)

print("=" * 60)
print("          FEATURE IMPORTANCE (Correlation-Based)")
print("=" * 60)
for name, imp in sorted_importance:
    bar = '█' * int(abs(imp) * 30)
    sign = '+' if imp > 0 else '-'
    print(f"  {name:30s}: {sign}{abs(imp):.3f} {bar}")

In [ ]:
# Feature Importance Bar Chart
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Horizontal bar chart
names = [x[0] for x in sorted_importance]
values = [x[1] for x in sorted_importance]
colors = ['#2ecc71' if v > 0 else '#e74c3c' for v in values]

ax = axes[0]
bars = ax.barh(range(len(names)), values, color=colors, edgecolor='white')
ax.set_yticks(range(len(names)))
ax.set_yticklabels(names)
ax.set_xlabel('Correlation with Max Range', fontsize=12)
ax.set_title('Feature Importance (Correlation)', fontsize=14, fontweight='bold')
ax.axvline(x=0, color='black', linewidth=0.5)

# Absolute importance
ax = axes[1]
abs_values = [abs(v) for v in values]
ax.barh(range(len(names)), abs_values, color='#3498db', edgecolor='white')
ax.set_yticks(range(len(names)))
ax.set_yticklabels(names)
ax.set_xlabel('Absolute Correlation', fontsize=12)
ax.set_title('Feature Importance (Absolute)', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('../reports/figures/feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n✅ Feature importance visualization saved!")

## 🔧 Step 1: Prepare Data for Training

In [ ]:
# Select features for model
# Use top correlated features
selected_features = [name for name, val in sorted_importance if abs(val) > 0.1]
print(f"Selected {len(selected_features)} features with |correlation| > 0.1")

# Prepare X and y
X = df[selected_features].values
y = df[target].values.reshape(-1, 1)

print(f"\nX shape: {X.shape}")
print(f"y shape: {y.shape}")

In [ ]:
# Standardize features (important for gradient descent!)
class StandardScaler:
    def __init__(self):
        self.mean = None
        self.std = None
    
    def fit(self, X):
        self.mean = np.mean(X, axis=0)
        self.std = np.std(X, axis=0)
        self.std[self.std == 0] = 1  # Avoid division by zero
        return self
    
    def transform(self, X):
        return (X - self.mean) / self.std
    
    def fit_transform(self, X):
        self.fit(X)
        return self.transform(X)


# Train/test split
def train_test_split(X, y, test_size=0.2, random_state=42):
    np.random.seed(random_state)
    n = len(X)
    indices = np.random.permutation(n)
    test_size = int(n * test_size)
    test_idx, train_idx = indices[:test_size], indices[test_size:]
    return X[train_idx], X[test_idx], y[train_idx], y[test_idx]


# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

# Standardize
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Add bias term (column of 1s)
X_train_scaled = np.c_[np.ones(X_train_scaled.shape[0]), X_train_scaled]
X_test_scaled = np.c_[np.ones(X_test_scaled.shape[0]), X_test_scaled]

print(f"Training set: X={X_train_scaled.shape}, y={y_train.shape}")
print(f"Test set: X={X_test_scaled.shape}, y={y_test.shape}")

## 🧮 Task 6: Linear Regression from Scratch

In [ ]:
class LinearRegression:
    """
    Linear Regression implemented from scratch using NumPy.
    
    Supports:
    - Gradient Descent optimization
    - Normal Equation (closed-form solution)
    
    Mathematical Model:
    h(X) = X @ θ
    J(θ) = (1/2m) * Σ(h(X) - y)²
    ∇J(θ) = (1/m) * X.T @ (X @ θ - y)
    """
    
    def __init__(self, learning_rate=0.01, n_iterations=1000, method='gradient_descent'):
        """
        Parameters:
        -----------
        learning_rate : float
            Step size for gradient descent (alpha)
        n_iterations : int
            Number of gradient descent iterations
        method : str
            'gradient_descent' or 'normal_equation'
        """
        self.learning_rate = learning_rate
        self.n_iterations = n_iterations
        self.method = method
        self.theta = None
        self.cost_history = []
    
    def _compute_cost(self, X, y):
        """
        Compute the MSE cost function.
        
        J(θ) = (1/2m) * (Xθ - y)ᵀ(Xθ - y)
        """
        m = len(y)
        predictions = X @ self.theta
        errors = predictions - y
        cost = (1 / (2 * m)) * np.sum(errors ** 2)
        return cost
    
    def _compute_gradient(self, X, y):
        """
        Compute gradient of cost function.
        
        ∇J(θ) = (1/m) * Xᵀ(Xθ - y)
        """
        m = len(y)
        predictions = X @ self.theta
        errors = predictions - y
        gradient = (1 / m) * (X.T @ errors)
        return gradient
    
    def _gradient_descent(self, X, y):
        """
        Perform gradient descent optimization.
        
        θ := θ - α * ∇J(θ)
        """
        for i in range(self.n_iterations):
            # Compute gradient
            gradient = self._compute_gradient(X, y)
            
            # Update weights
            self.theta = self.theta - self.learning_rate * gradient
            
            # Track cost
            cost = self._compute_cost(X, y)
            self.cost_history.append(cost)
            
            # Early stopping if cost is not decreasing
            if i > 0 and abs(self.cost_history[-2] - cost) < 1e-9:
                print(f"   Converged at iteration {i}")
                break
    
    def _normal_equation(self, X, y):
        """
        Closed-form solution.
        
        θ = (XᵀX)⁻¹ Xᵀy
        """
        self.theta = np.linalg.pinv(X.T @ X) @ X.T @ y
    
    def fit(self, X, y):
        """
        Train the model.
        """
        # Initialize weights randomly
        n_features = X.shape[1]
        self.theta = np.random.randn(n_features, 1) * 0.01
        self.cost_history = []
        
        print(f"Training Linear Regression ({self.method})...")
        print(f"   Features: {n_features}")
        print(f"   Samples: {len(y)}")
        
        if self.method == 'gradient_descent':
            print(f"   Learning rate: {self.learning_rate}")
            print(f"   Max iterations: {self.n_iterations}")
            self._gradient_descent(X, y)
        else:
            self._normal_equation(X, y)
        
        print(f"\n✅ Training complete!")
        if self.cost_history:
            print(f"   Final cost: {self.cost_history[-1]:.6f}")
        
        return self
    
    def predict(self, X):
        """
        Make predictions.
        
        ŷ = Xθ
        """
        return X @ self.theta
    
    def get_weights(self):
        """Return learned weights."""
        return self.theta


print("✅ LinearRegression class defined!")

In [ ]:
# Train model with Gradient Descent
model_gd = LinearRegression(
    learning_rate=0.1,
    n_iterations=1000,
    method='gradient_descent'
)
model_gd.fit(X_train_scaled, y_train)

# Also train with Normal Equation for comparison
model_ne = LinearRegression(method='normal_equation')
model_ne.fit(X_train_scaled, y_train)

## 📉 Visualize Training Process

In [ ]:
# Plot cost function convergence
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Full cost history
ax = axes[0]
ax.plot(model_gd.cost_history, color='#e74c3c', linewidth=2)
ax.set_xlabel('Iteration', fontsize=12)
ax.set_ylabel('Cost J(θ)', fontsize=12)
ax.set_title('Gradient Descent: Cost Function Convergence', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3)

# Zoomed view (first 100 iterations)
ax = axes[1]
ax.plot(model_gd.cost_history[:100], color='#3498db', linewidth=2)
ax.set_xlabel('Iteration', fontsize=12)
ax.set_ylabel('Cost J(θ)', fontsize=12)
ax.set_title('First 100 Iterations (Zoomed)', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../reports/figures/gradient_descent_convergence.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\nInitial cost: {model_gd.cost_history[0]:.4f}")
print(f"Final cost: {model_gd.cost_history[-1]:.4f}")
print(f"Cost reduction: {(1 - model_gd.cost_history[-1]/model_gd.cost_history[0])*100:.2f}%")

## 📊 Model Evaluation

In [ ]:
class RegressionMetrics:
    """
    Evaluation metrics for regression models.
    All implemented from scratch using NumPy.
    """
    
    @staticmethod
    def mse(y_true, y_pred):
        """
        Mean Squared Error.
        MSE = (1/m) * Σ(y - ŷ)²
        """
        return np.mean((y_true - y_pred) ** 2)
    
    @staticmethod
    def rmse(y_true, y_pred):
        """
        Root Mean Squared Error.
        RMSE = √MSE
        """
        return np.sqrt(RegressionMetrics.mse(y_true, y_pred))
    
    @staticmethod
    def mae(y_true, y_pred):
        """
        Mean Absolute Error.
        MAE = (1/m) * Σ|y - ŷ|
        """
        return np.mean(np.abs(y_true - y_pred))
    
    @staticmethod
    def r2_score(y_true, y_pred):
        """
        R² Score (Coefficient of Determination).
        R² = 1 - SS_res / SS_tot
        """
        ss_res = np.sum((y_true - y_pred) ** 2)
        ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
        return 1 - (ss_res / ss_tot)
    
    @staticmethod
    def evaluate(y_true, y_pred, name='Model'):
        """Print comprehensive evaluation report."""
        mse = RegressionMetrics.mse(y_true, y_pred)
        rmse = RegressionMetrics.rmse(y_true, y_pred)
        mae = RegressionMetrics.mae(y_true, y_pred)
        r2 = RegressionMetrics.r2_score(y_true, y_pred)
        
        print(f"\n{'='*50}")
        print(f"  {name} - Evaluation Metrics")
        print(f"{'='*50}")
        print(f"  MSE:  {mse:.4f}")
        print(f"  RMSE: {rmse:.4f}")
        print(f"  MAE:  {mae:.4f}")
        print(f"  R²:   {r2:.4f} ({r2*100:.2f}%)")
        print(f"{'='*50}")
        
        return {'mse': mse, 'rmse': rmse, 'mae': mae, 'r2': r2}


# Make predictions
y_train_pred = model_gd.predict(X_train_scaled)
y_test_pred = model_gd.predict(X_test_scaled)

# Evaluate
train_metrics = RegressionMetrics.evaluate(y_train, y_train_pred, 'Training Set')
test_metrics = RegressionMetrics.evaluate(y_test, y_test_pred, 'Test Set')

In [ ]:
# Prediction visualizations
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# 1. Actual vs Predicted (Training)
ax = axes[0, 0]
ax.scatter(y_train, y_train_pred, alpha=0.5, s=10, c='#3498db')
ax.plot([y_train.min(), y_train.max()], [y_train.min(), y_train.max()], 
        'r--', linewidth=2, label='Perfect Prediction')
ax.set_xlabel('Actual Max Range (km)', fontsize=11)
ax.set_ylabel('Predicted Max Range (km)', fontsize=11)
ax.set_title(f'Training: Actual vs Predicted (R²={train_metrics["r2"]:.3f})', fontweight='bold')
ax.legend()

# 2. Actual vs Predicted (Test)
ax = axes[0, 1]
ax.scatter(y_test, y_test_pred, alpha=0.5, s=10, c='#2ecc71')
ax.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 
        'r--', linewidth=2, label='Perfect Prediction')
ax.set_xlabel('Actual Max Range (km)', fontsize=11)
ax.set_ylabel('Predicted Max Range (km)', fontsize=11)
ax.set_title(f'Test: Actual vs Predicted (R²={test_metrics["r2"]:.3f})', fontweight='bold')
ax.legend()

# 3. Residuals Distribution
ax = axes[1, 0]
residuals = (y_test - y_test_pred).flatten()
ax.hist(residuals, bins=50, color='#9b59b6', edgecolor='white', alpha=0.7)
ax.axvline(x=0, color='red', linestyle='--', linewidth=2)
ax.set_xlabel('Residual (Actual - Predicted)', fontsize=11)
ax.set_ylabel('Frequency', fontsize=11)
ax.set_title('Residuals Distribution (should be centered at 0)', fontweight='bold')

# 4. Residuals vs Predicted
ax = axes[1, 1]
ax.scatter(y_test_pred, residuals, alpha=0.5, s=10, c='#e74c3c')
ax.axhline(y=0, color='black', linestyle='-', linewidth=1)
ax.set_xlabel('Predicted Max Range (km)', fontsize=11)
ax.set_ylabel('Residual', fontsize=11)
ax.set_title('Residuals vs Predicted (check for patterns)', fontweight='bold')

plt.tight_layout()
plt.savefig('../reports/figures/linear_regression_evaluation.png', dpi=150, bbox_inches='tight')
plt.show()

## 🔍 Feature Weights Analysis

In [ ]:
# Analyze learned weights
weights = model_gd.get_weights().flatten()
feature_names = ['bias'] + selected_features

# Create weight DataFrame
weight_df = pd.DataFrame({
    'Feature': feature_names,
    'Weight': weights,
    'Abs_Weight': np.abs(weights)
}).sort_values('Abs_Weight', ascending=False)

print("\n" + "=" * 50)
print("         LEARNED FEATURE WEIGHTS")
print("=" * 50)
for _, row in weight_df.iterrows():
    sign = '+' if row['Weight'] > 0 else '-'
    print(f"  {row['Feature']:30s}: {sign}{abs(row['Weight']):.4f}")

In [ ]:
# Visualize weights
plt.figure(figsize=(10, 8))

# Exclude bias for visualization
weight_df_no_bias = weight_df[weight_df['Feature'] != 'bias'].copy()
colors = ['#2ecc71' if w > 0 else '#e74c3c' for w in weight_df_no_bias['Weight']]

plt.barh(range(len(weight_df_no_bias)), weight_df_no_bias['Weight'], color=colors, edgecolor='white')
plt.yticks(range(len(weight_df_no_bias)), weight_df_no_bias['Feature'])
plt.xlabel('Weight Value', fontsize=12)
plt.title('Linear Regression: Learned Feature Weights', fontsize=14, fontweight='bold')
plt.axvline(x=0, color='black', linewidth=0.5)
plt.tight_layout()
plt.savefig('../reports/figures/feature_weights.png', dpi=150, bbox_inches='tight')
plt.show()

## 💾 Save Model

In [ ]:
# Save model weights
model_data = {
    'weights': model_gd.get_weights().flatten().tolist(),
    'feature_names': feature_names,
    'scaler_mean': scaler.mean.tolist(),
    'scaler_std': scaler.std.tolist(),
    'metrics': {
        'train': train_metrics,
        'test': test_metrics
    }
}

with open('../models/linear_regression_model.json', 'w') as f:
    json.dump(model_data, f, indent=2)

# Also save as numpy
np.save('../models/linear_regression_weights.npy', model_gd.get_weights())

print("\n✅ Model saved!")
print("   - models/linear_regression_model.json")
print("   - models/linear_regression_weights.npy")

---

## 📚 Learning Resources

### Blogs & Articles
1. **Linear Regression from Scratch**: [A Complete Guide](https://towardsdatascience.com/linear-regression-from-scratch-with-numpy-implementation-finally-388c007e1b96)
2. **Gradient Descent Explained**: [Visual Introduction](https://ml-cheatsheet.readthedocs.io/en/latest/gradient_descent.html)
3. **R² Score Explained**: [Understanding R-squared](https://statisticsbyjim.com/regression/interpret-r-squared-regression/)

### YouTube Videos
1. **Andrew Ng - Linear Regression**: [Stanford ML Course](https://www.youtube.com/watch?v=4b4MUYve_U8)
2. **3Blue1Brown - Gradient Descent**: [Visual Explanation](https://www.youtube.com/watch?v=IHZwWFHWa-w)

### Research Papers
1. Cauchy (1847) - Original gradient descent method
2. Gauss, Legendre - Least squares method development

---

## ✅ Tasks 5 & 6 Complete!

### What We Accomplished:
- ✅ Visualized feature importance using correlation
- ✅ Built Linear Regression from scratch with NumPy
- ✅ Implemented gradient descent optimization
- ✅ Implemented normal equation solution
- ✅ Created comprehensive evaluation metrics (MSE, RMSE, MAE, R²)
- ✅ Visualized training convergence and predictions
- ✅ Analyzed learned feature weights

### Model Performance:
| Set | R² Score |
|-----|----------|
| Training | See output above |
| Test | See output above |

### Next Tasks: Polynomial Regression & Logistic Regression
We'll improve the range estimator and build the success predictor!

---